In [ ]:
# Download unbinned data GRB if required
from cosipy.util import fetch_wasabi_file
#fetch_wasabi_file('COSI-SMEX/DC3/Data/Sources/GRB_bn110605183_3months_unbinned_data_filtered_with_SAAcut.fits.gz')
#fetch_wasabi_file('COSI-SMEX/DC3/Data/Backgrounds/Ge/Total_BG_with_SAAcomponent_3months_unbinned_data_filtered_with_SAAcut.fits.gz')

In [ ]:
from astropy.coordinates import SkyCoord
import astropy.units as u
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import numpy as np

In [ ]:
from astropy.io import fits

data_dir = "/data/test_newrepo/"

# Open the compressed FITS file
hdul = fits.open(data_dir+"GRB_bn110605183_3months_unbinned_data_filtered_with_SAAcut.fits.gz")

# Display summary of the file structure
hdul.info()

In [ ]:
table_hdu = hdul[1]  # or use the HDU name, e.g. hdul['DATA']
data_grb = table_hdu.data

In [ ]:
data_grb.columns

In [ ]:
#true_l = 221
#true_b = -19

true_l = 41.64
true_b = 19.24

In [ ]:
def calculate_arm(true_l,true_b,event_l,event_b,phi):
    
    true_coord = SkyCoord(l=true_l * u.deg, b=true_b * u.deg, frame='galactic')
    coords = SkyCoord(l=event_l * u.deg, b=event_b * u.deg, frame='galactic')
    
    # Compute angular separation
    sep = coords.separation(true_coord)
    
    # Convert to degrees
    angular_distances = sep.deg

    ARM = angular_distances - phi 

    return ARM
    

In [ ]:
ARM = calculate_arm(true_l,true_b,data_grb['Chi galactic'],data_grb['Psi galactic'],np.rad2deg(data_grb['Phi']))

In [ ]:
arm_min, arm_max = -15, 15


plt.hist(ARM, bins=30, color='skyblue', edgecolor='black')

plt.xlabel("ARM")
plt.ylabel("Counts")

plt.axvline(x=arm_min, color='red', linestyle='--', linewidth=2, label='Lower limit')
plt.axvline(x=arm_max, color='red', linestyle='--', linewidth=2, label='Upper limit')

plt.show()

In [ ]:
times = data_grb['TimeTags']

tmin = times.min()
tmax = times.max()

print(tmin)
print(tmax)
print(tmax-tmin)

In [ ]:
bin_size = 1.0  # secondi

bins = np.arange(tmin, tmax + bin_size, bin_size)

# --- Conta gli eventi per bin ---
counts, bin_edges = np.histogram(times, bins=bins)

# --- Calcola il tempo centrale di ciascun bin (per comodità) ---
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# --- Step plot ---
plt.figure(figsize=(8,4))
plt.step(bin_centers, counts, where='mid', color='royalblue')

plt.xlabel("Time [s]")
plt.ylabel("Counts")
plt.title(f"Light curve ({bin_size:.1f} s per bin)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


# Background

In [ ]:
# Open the compressed FITS file
hdul_bkg = fits.open(data_dir+"Total_BG_with_SAAcomponent_3months_unbinned_data_filtered_with_SAAcut.fits")

# Display summary of the file structure
hdul_bkg.info()

In [ ]:
hdul_bkg[1].data.columns

In [ ]:
# Filter for GRB time
evt_hdu_bkg = hdul_bkg[1]                      
data_bkg = evt_hdu_bkg.data                    
times_bkg = data_bkg['TimeTags']                   

mask = (times_bkg >= tmin) & (times_bkg < tmax)
data_bkg_filt = data_bkg[mask]

times_bkg_filtered = data_bkg_filt['TimeTags']


In [ ]:
ARM_bkg = calculate_arm(true_l,true_b,data_bkg_filt['Chi galactic'],data_bkg_filt['Psi galactic'],np.rad2deg(data_bkg_filt['Phi']))

In [ ]:
plt.hist(ARM_bkg, bins=30, color='skyblue', edgecolor='black')

plt.axvline(x=arm_min, color='red', linestyle='--', linewidth=2, label='Lower limit')
plt.axvline(x=arm_max, color='red', linestyle='--', linewidth=2, label='Upper limit')


plt.xlabel("ARM")
plt.ylabel("Counts")

plt.show()

In [ ]:
bins = np.arange(tmin, tmax + bin_size, bin_size)

# --- Conta gli eventi per bin ---
counts, bin_edges = np.histogram(times_bkg_filtered, bins=bins)

# --- Calcola il tempo centrale di ciascun bin (per comodità) ---
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# --- Step plot ---
plt.figure(figsize=(8,4))
plt.step(bin_centers, counts, where='mid', color='royalblue')

plt.xlabel("Tempo [s]")
plt.ylabel("Numero di eventi per bin")
plt.title(f"Light curve ({bin_size:.1f} s per bin)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


# Sum GRB and Background

In [ ]:
#filter source data with ARM

mask_arm = (ARM >= arm_min) & (ARM <= arm_max)

data_grb_filtered = data_grb[mask_arm]

In [ ]:
#filter background data with ARM

mask_arm_bkg = (ARM_bkg >= arm_min) & (ARM_bkg <= arm_max)

data_bkg_filtered = data_bkg_filt[mask_arm_bkg]

In [ ]:
data_filtered_summed = data_combined = np.hstack([data_grb_filtered, data_bkg_filtered])

In [ ]:
bins = np.arange(tmin, tmax + bin_size, bin_size)

counts, bin_edges = np.histogram(data_filtered_summed['TimeTags'], bins=bins)

bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# --- Step plot ---
plt.figure(figsize=(8,4))
plt.step(bin_centers, counts, where='mid', color='royalblue')

plt.xlabel("Time [s]")
plt.ylabel("Counts")
plt.title(f"GRB+BKG LC ({bin_size:.1f} s bin)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
bins = np.arange(tmin, tmax + bin_size, bin_size)

counts, bin_edges = np.histogram(data_grb_filtered['TimeTags'], bins=bins)

bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

plt.figure(figsize=(8,4))
plt.step(bin_centers, counts, where='mid', color='royalblue')

plt.xlabel("Time [s]")
plt.ylabel("Counts")
plt.title(f"GRB LC ({bin_size:.1f} s bin)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
bins = np.arange(tmin, tmax + bin_size, bin_size)

counts, bin_edges = np.histogram(data_bkg_filtered['TimeTags'], bins=bins)

bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

plt.figure(figsize=(8,4))
plt.step(bin_centers, counts, where='mid', color='royalblue')

plt.xlabel("Time [s]")
plt.ylabel("Counts")
plt.title(f"BKG ({bin_size:.1f} s  bin)")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Legge il file (sostituisci 'file.dat' col nome del tuo file)
df = pd.read_csv(data_dir+"bn110605183_lightcurve.dat", sep=r"\s+", header=None, comment="#")

# Assegna nomi alle colonne
df.columns = ["flag", "time", "value"]

# Se la prima colonna ("DP") non serve, la scartiamo
df = df.drop(columns=["flag"])

# Plot
plt.figure(figsize=(8,5))
plt.plot(df["time"], df["value"], marker='.', linestyle='-')
plt.xlabel("Time (s)")
plt.ylabel("Value")
plt.title("LC shape")
plt.grid(True)
plt.show()